# Migraine HD-EEG — Paper-Faithful Preprocessing

This notebook implements the **original paper** preprocessing pipeline  
(Chamanzar, Haigh, Grover & Behrmann, 2020) using MNE-Python.

### Key differences vs. `preprocess_migraine.py` (LEMON-style)

| Step | LEMON-style (`preprocess_migraine.py`) | **Paper-faithful (this notebook)** |
|------|----------------------------------------|--------------------------------------|
| Reference | Average reference | **Mastoid (M1+M2)** |
| High-pass | 1.0 Hz FIR | **0.1 Hz IIR Butterworth** |
| Bad ch. before ICA? | ✓ | ✓ (required by paper) |
| ICA labelling | ICLabel (auto-ML) | **find_bads_eog / find_bads_ecg** (physiological channels) |
| Epoch length | 4 s, 50 % overlap | **2 s, 0 % overlap (rest) / event-locked (task)** |
| Task epochs | Fixed-length | **Event-locked 0–2 s post-stimulus** |
| Frequency bands | Broadband only | **5 bands: δ θ α β γ saved separately** |
| Resampling | 250 Hz | **512 Hz kept** (no resampling unless chosen) |

### Event codes (from Status channel)
- `1` → SSAEP/SSVEP stimulus frequency-1 onset (100 trials)  
- `2` → SSAEP/SSVEP stimulus frequency-2 onset (100 trials)  
- `11/22/30` → response / session markers (ignored in epoching)

### Outputs under `data/MIGRAINE_paper_preprocessed/`
```
resting/   {subject}_broadband.npy   (n_epochs, 128, 1024)  @ 512 Hz
           {subject}_delta.npy  ..theta..alpha..beta..gamma.npy
ssaep/     {subject}_broadband.npy   (n_epochs, 128, 1024)  event-locked
           {subject}_delta.npy  ...
ssvep/     ...
channel_names.txt
preprocessing_summary.csv
labels.csv
dataset_{condition}.npz    → X, y, groups, ch_names, sfreq (broadband only)
```

## 1 · Imports & environment

In [ ]:
import os, re, glob, time, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import mne
from mne.preprocessing import ICA, create_eog_epochs, create_ecg_epochs

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore', message='.*encountered in matmul.*')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='pyprep')
%matplotlib inline
print(f'MNE {mne.__version__}  NumPy {np.__version__}')

## 2 · Parameters

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = os.getcwd()
DATASET_DIR = os.path.join(BASE_DIR, 'Dataset')
OUTPUT_DIR  = os.path.join(BASE_DIR, 'data', 'MIGRAINE_paper_preprocessed')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Filtering (paper-faithful) ────────────────────────────────────────────────
HIGHPASS_FREQ  = 0.1     # paper: 0.1 Hz
LOWPASS_FREQ   = 100.0   # paper: 100 Hz
FILTER_METHOD  = 'iir'
FILTER_PARAMS  = dict(order=4, ftype='butter')   # zero-phase Butterworth

# ── Sampling ─────────────────────────────────────────────────────────────────
# The paper does NOT resample (stays at 512 Hz).  Set RESAMPLE=True to
# downsample to 256 Hz for lighter storage / faster training.
RESAMPLE       = False
TARGET_SFREQ   = 256     # only used if RESAMPLE=True

# ── Epoch settings ────────────────────────────────────────────────────────────
EPOCH_DURATION_S = 2.0   # paper: 2 s
# Resting: non-overlapping (overlap=0)
# Task:    event-locked 0 → +2 s after stimulus onset
TASK_TMIN, TASK_TMAX = 0.0, 2.0
TASK_EVENT_IDS = {'freq1': 1, 'freq2': 2}  # stimulus onset triggers

AMPLITUDE_REJECT_UV = 150e-6   # slightly tighter than broadband: 150 µV p2p

# ── ICA ──────────────────────────────────────────────────────────────────────
ICA_N_COMPONENTS = 64     # PCA whitening keeps compute tractable
ICA_METHOD       = 'infomax'
ICA_FIT_PARAMS   = dict(extended=True)
RANDOM_STATE     = 42

# ── Bad-channel detection ─────────────────────────────────────────────────────
# Automated stand-in for paper's "visual inspection".
# Channels with std < FLAT_STD or > NOISY_STD_MULT×median are flagged.
FLAT_STD_UV      = 0.5    # µV – effectively dead channel
NOISY_STD_MULT   = 5.0    # × median std
MAX_BAD_FRAC     = 0.15   # never interpolate more than 15% of channels

# ── Frequency bands (paper Table 1) ──────────────────────────────────────────
BANDS = {
    'delta': (0.5, 3),
    'theta': (4,   7),
    'alpha': (8,  12),
    'beta' : (12, 30),
    'gamma': (30,100),
}

# ── Channel roles (BioSemi ActiveTwo in this dataset) ────────────────────────
EOG_CHANNELS  = ['LO1', 'LO2', 'IO1', 'IO2', 'SO1']   # periocular
ECG_CHANNEL   = 'ECG'
MASTOID_CHNLS = ['M1', 'M2']                           # reference electrodes
AUX_CHANNELS  = ['GSR1','GSR2','Erg1','Erg2','Resp','Plet','Temp']  # drop
STIM_CHANNEL  = 'Status'

# ── Subjects excluded from original study analysis (still preprocessed) ───────
ORIG_EXCLUDED = {'M2_1','M6_1','M18_1','M13_2','C2','C6','C12'}

CONDITION_KEYS = {'resting': 'rest', 'ssaep': 'ssaep', 'ssvep': 'ssvep'}

sfreq_out = TARGET_SFREQ if RESAMPLE else 512
n_samples  = int(EPOCH_DURATION_S * sfreq_out)
print(f'Output: {sfreq_out} Hz | {n_samples} samples/epoch | epoch={EPOCH_DURATION_S}s')
print(f'Bands : {list(BANDS.keys())}')

## 3 · Helper functions

In [ ]:
# ─── File discovery ───────────────────────────────────────────────────────────
def discover_subjects(dataset_dir):
    """Return {subject_id: {condition: filepath}} handling nested extraction layout."""
    subjects = {}
    for sub in sorted(d for d in os.listdir(dataset_dir)
                      if os.path.isdir(os.path.join(dataset_dir, d))
                      and re.match(r'^[CM]\d+', d)):
        folder = os.path.join(dataset_dir, sub)
        bdfs = [
            f for f in glob.glob(os.path.join(folder, '**', '*.bdf'), recursive=True)
            if '__MACOSX' not in f
        ]
        cond_files = {}
        for cond, key in CONDITION_KEYS.items():
            hits = [f for f in bdfs if key in os.path.basename(f).lower()]
            if hits:
                hits.sort(key=lambda f: ('_2' in os.path.basename(f),
                                         len(os.path.basename(f))))
                cond_files[cond] = hits[0]
        subjects[sub] = cond_files
    return subjects


def subject_group(subject_id):
    if subject_id.startswith('M'):
        aura = 'with_aura' if subject_id.endswith('_2') else 'without_aura'
        return 'migraine', 1, aura
    return 'control', 0, 'n/a'


# ─── Channel setup ────────────────────────────────────────────────────────────
def setup_channels(raw):
    """Assign correct channel types, drop pure-auxiliary channels."""
    # Drop channels we don't need at all
    drop = [c for c in AUX_CHANNELS + [STIM_CHANNEL] if c in raw.ch_names]
    raw.drop_channels(drop)

    type_map = {}
    for ch in EOG_CHANNELS:
        if ch in raw.ch_names:
            type_map[ch] = 'eog'
    if ECG_CHANNEL in raw.ch_names:
        type_map[ECG_CHANNEL] = 'ecg'
    # Mastoids are kept as EEG so they can be used as reference
    for ch in MASTOID_CHNLS:
        if ch in raw.ch_names:
            type_map[ch] = 'eeg'
    raw.set_channel_types(type_map)

    # Set montage for the 128 scalp channels (10-05 names match exactly)
    montage = mne.channels.make_standard_montage('standard_1005')
    raw.set_montage(montage, on_missing='ignore', verbose=False)
    return raw


# ─── Bad-channel detection ────────────────────────────────────────────────────
def detect_bad_channels_amplitude(raw, manual_bads=None):
    """
    Automated stand-in for the paper's visual inspection.
    Flags channels that are flat (std < FLAT_STD_UV) or
    excessively noisy (std > NOISY_STD_MULT × median std).
    Also incorporates any manually specified bad channels.
    """
    eeg_picks = mne.pick_types(raw.info, eeg=True, exclude=[])
    data_uv   = raw.get_data(picks=eeg_picks) * 1e6
    stds      = data_uv.std(axis=1)
    median_s  = np.median(stds)

    flat_mask  = stds < FLAT_STD_UV
    noisy_mask = stds > NOISY_STD_MULT * median_s
    flagged    = np.array(raw.ch_names)[eeg_picks][flat_mask | noisy_mask].tolist()

    # Also try PyPREP nan/flat/deviation (no RANSAC – geometry approximate)
    try:
        from pyprep.find_noisy_channels import NoisyChannels
        rp = raw.copy().pick('eeg')
        nc = NoisyChannels(rp, random_state=RANDOM_STATE)
        nc.find_bad_by_nan_flat()
        nc.find_bad_by_deviation()
        flagged = sorted(set(flagged) | set(nc.bad_by_nan) |
                         set(nc.bad_by_flat) | set(nc.bad_by_deviation))
    except Exception:
        pass

    if manual_bads:
        flagged = sorted(set(flagged) | set(manual_bads))

    # Cap at MAX_BAD_FRAC of EEG channels
    n_eeg  = len(eeg_picks)
    cap    = int(MAX_BAD_FRAC * n_eeg)
    capped = len(flagged) > cap
    if capped:
        # Keep only flat + manual (most trustworthy)
        safe = [raw.ch_names[p] for p in eeg_picks[flat_mask]]
        if manual_bads:
            safe = sorted(set(safe) | set(manual_bads))
        flagged = safe[:cap]

    return flagged, capped


# ─── Band filtering ───────────────────────────────────────────────────────────
def filter_band(epochs, l_freq, h_freq):
    """Return a copy of epochs bandpass filtered with IIR Butterworth."""
    return epochs.copy().filter(
        l_freq=l_freq, h_freq=h_freq,
        method=FILTER_METHOD,
        iir_params=FILTER_PARAMS,
        picks='eeg', verbose=False,
    )


print('Helper functions defined.')

## 4 · Core preprocessing function

In [ ]:
def preprocess_paper(
    bdf_file,
    subject_id,
    condition,           # 'resting' | 'ssaep' | 'ssvep'
    output_dir,
    manual_bads=None,    # list of channel names if you want to override
    overwrite=False,
    return_epochs=False, # set True in demo to get the objects back
):
    """
    Paper-faithful preprocessing pipeline for one BioSemi BDF recording.

    Steps (matching Chamanzar et al. 2020):
      1. Load + set channel types (EOG, ECG, mastoids, scalp EEG)
      2. Re-reference to M1+M2 (mastoid reference)
      3. Zero-phase Butterworth bandpass 0.1 – 100 Hz
      4. Detect & interpolate bad channels  ← BEFORE ICA
      5. ICA (extended infomax, 64 PCA comps) → find_bads_eog / find_bads_ecg
      6. Epoch: 2 s event-locked (task) or 2 s non-overlapping (resting)
      7. Reject epochs > 150 µV p2p
      8. Save broadband + 5 frequency-band epoch arrays
    """
    cond_dir  = os.path.join(output_dir, condition)
    os.makedirs(cond_dir, exist_ok=True)

    bb_path = os.path.join(cond_dir, f'{subject_id}_broadband.npy')
    if os.path.exists(bb_path) and not overwrite:
        return {'subject': subject_id, 'condition': condition,
                'status': 'skipped_existing'}

    result = {'subject': subject_id, 'condition': condition, 'status': 'success'}
    group, label, aura = subject_group(subject_id)
    result.update(group=group, label=label, aura=aura,
                  excluded_in_orig_study=subject_id in ORIG_EXCLUDED)

    try:
        # ── 1. Load ───────────────────────────────────────────────────────────
        raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=False)
        raw_before = raw.copy()   # kept for QC plot if needed

        raw = setup_channels(raw)
        result['n_eeg_in'] = len(mne.pick_types(raw.info, eeg=True))

        # ── 2. Mastoid re-reference ───────────────────────────────────────────
        available_mastoids = [c for c in MASTOID_CHNLS if c in raw.ch_names]
        if not available_mastoids:
            raise RuntimeError('No mastoid channels found; cannot re-reference.')
        raw.set_eeg_reference(ref_channels=available_mastoids,
                              projection=False, verbose=False)
        result['mastoid_ref'] = ','.join(available_mastoids)

        # ── 3. Bandpass 0.1 – 100 Hz (IIR Butterworth, zero-phase) ───────────
        raw.filter(
            l_freq=HIGHPASS_FREQ, h_freq=LOWPASS_FREQ,
            method=FILTER_METHOD,
            iir_params=FILTER_PARAMS,
            picks='eeg', verbose=False,
        )

        # Optional resample
        if RESAMPLE:
            raw.resample(sfreq=TARGET_SFREQ, npad='auto', n_jobs=1, verbose=False)

        # ── 4. Bad channel detection & interpolation  (BEFORE ICA) ────────────
        bad_chs, capped = detect_bad_channels_amplitude(raw, manual_bads)
        raw.info['bads'] = bad_chs
        result['n_bad_channels'] = len(bad_chs)
        result['bad_channels']   = ','.join(bad_chs)
        result['flag_many_bads'] = capped

        if bad_chs:
            raw.interpolate_bads(reset_bads=True, verbose=False)

        # ── 5. ICA ─────────────────────────────────────────────────────────────
        n_eeg = len(mne.pick_types(raw.info, eeg=True))
        n_comp = min(ICA_N_COMPONENTS, n_eeg - 1)
        ica = ICA(
            n_components=n_comp,
            method=ICA_METHOD,
            fit_params=ICA_FIT_PARAMS,
            random_state=RANDOM_STATE,
            max_iter='auto',
            verbose=False,
        )
        ica.fit(raw, picks='eeg', verbose=False)

        exclude = []

        # Eye-blink / horizontal eye movement
        eog_present = [c for c in EOG_CHANNELS if c in raw.ch_names]
        if eog_present:
            for ch in eog_present:
                try:
                    idxs, _ = ica.find_bads_eog(raw, ch_name=ch, verbose=False)
                    exclude.extend(idxs)
                except Exception:
                    pass

        # Heartbeat
        if ECG_CHANNEL in raw.ch_names:
            try:
                idxs, _ = ica.find_bads_ecg(raw, ch_name=ECG_CHANNEL,
                                             method='correlation', verbose=False)
                exclude.extend(idxs)
            except Exception:
                pass

        ica.exclude = sorted(set(exclude))
        raw_clean   = ica.apply(raw.copy(), verbose=False)
        result['n_ica_components'] = n_comp
        result['n_rejected_ics']   = len(ica.exclude)
        result['rejected_ics']     = ','.join(map(str, ica.exclude))

        # Drop mastoids and EOG/ECG now – keep only the 128 scalp EEG channels
        keep_only_eeg = mne.pick_types(raw_clean.info, eeg=True,
                                       include=[], exclude=MASTOID_CHNLS)
        scalp_names   = [raw_clean.ch_names[i] for i in keep_only_eeg]
        raw_clean.pick(scalp_names)
        raw_clean.reorder_channels(sorted(raw_clean.ch_names))

        # ── 6. Epoching ────────────────────────────────────────────────────────
        if condition == 'resting':
            # 2 s non-overlapping fixed windows
            epochs = mne.make_fixed_length_epochs(
                raw_clean, duration=EPOCH_DURATION_S,
                overlap=0.0, preload=True, verbose=False,
            )
        else:
            # Event-locked: re-find events on the ORIGINAL (pre-clean) raw
            # because Status channel was dropped from raw_clean.
            raw_stim = mne.io.read_raw_bdf(bdf_file, preload=False,
                                            verbose=False)
            events = mne.find_events(raw_stim, stim_channel=STIM_CHANNEL,
                                     verbose=False)
            # Keep only the two stimulus-onset codes
            mask   = np.isin(events[:, 2], list(TASK_EVENT_IDS.values()))
            events = events[mask]
            del raw_stim

            if len(events) == 0:
                raise RuntimeError('No stimulus events found in Status channel.')

            # Align event sample indices to the resampled timeline if needed
            if RESAMPLE:
                scale = TARGET_SFREQ / 512.0
                events[:, 0] = (events[:, 0] * scale).astype(int)

            epochs = mne.Epochs(
                raw_clean, events,
                event_id=TASK_EVENT_IDS,
                tmin=TASK_TMIN, tmax=TASK_TMAX - (1.0 / raw_clean.info['sfreq']),
                baseline=None,
                preload=True,
                verbose=False,
            )
            result['n_task_events'] = len(events)

        # ── 7. Reject noisy epochs ─────────────────────────────────────────────
        n_before = len(epochs)
        epochs.drop_bad(reject=dict(eeg=AMPLITUDE_REJECT_UV), verbose=False)
        result['n_epochs_before'] = n_before
        result['n_epochs_after']  = len(epochs)
        result['epoch_keep_ratio']= round(len(epochs) / max(1, n_before), 3)

        if len(epochs) == 0:
            raise RuntimeError('All epochs rejected.')

        result['epoch_shape'] = epochs.get_data().shape

        # ── 8a. Save broadband ─────────────────────────────────────────────────
        X_bb = epochs.get_data().astype(np.float32)   # (n, 128, n_times)
        np.save(bb_path, X_bb)
        result['npy_broadband'] = bb_path

        # ── 8b. Save channel names (once) ─────────────────────────────────────
        ch_path = os.path.join(output_dir, 'channel_names.txt')
        if not os.path.exists(ch_path):
            with open(ch_path, 'w') as fh:
                fh.write('\n'.join(epochs.ch_names))

        # ── 8c. Save per-band epochs ───────────────────────────────────────────
        band_paths = {}
        for band, (lf, hf) in BANDS.items():
            ep_band = filter_band(epochs, lf, hf)
            p = os.path.join(cond_dir, f'{subject_id}_{band}.npy')
            np.save(p, ep_band.get_data().astype(np.float32))
            band_paths[band] = p
        result['band_paths'] = band_paths

    except Exception as exc:
        result['status'] = 'failed'
        result['error']  = f'{type(exc).__name__}: {exc}'
        if return_epochs:
            return result, None, None
        return result

    if return_epochs:
        return result, raw_clean, epochs
    return result


print('Pipeline function defined.')

## 5 · Single-subject demo — C1 resting
Run this section to verify the pipeline on one recording before launching the full batch.

In [ ]:
DEMO_SUBJECT   = 'C1'
DEMO_CONDITION = 'resting'

subjects = discover_subjects(DATASET_DIR)
demo_bdf = subjects[DEMO_SUBJECT][DEMO_CONDITION]
print(f'Processing: {demo_bdf}')

t0 = time.time()
res, raw_clean_demo, epochs_demo = preprocess_paper(
    demo_bdf, DEMO_SUBJECT, DEMO_CONDITION, OUTPUT_DIR,
    overwrite=True, return_epochs=True,
)
print(f'Done in {time.time()-t0:.1f}s')
print('\n--- Result ---')
for k, v in res.items():
    if k != 'band_paths':
        print(f'  {k:25s}: {v}')

In [ ]:
# ── Epoch data shape & sanity check ──────────────────────────────────────────
if epochs_demo is not None:
    X = epochs_demo.get_data()
    print(f'Epoch array shape : {X.shape}  (n_epochs, n_ch, n_samples)')
    print(f'Any NaN/Inf       : {np.isnan(X).any()} / {np.isinf(X).any()}')
    print(f'Amplitude range µV: {X.min()*1e6:.1f}  to  {X.max()*1e6:.1f}')
    print(f'Mean |amplitude| µV: {np.abs(X).mean()*1e6:.3f}')

In [ ]:
# ── PSD of cleaned data ────────────────────────────────────────────────────
if raw_clean_demo is not None:
    fig = raw_clean_demo.compute_psd(fmin=0.5, fmax=100, picks='eeg', verbose=False).plot(
        show=False, average=True
    )
    fig.suptitle(f'{DEMO_SUBJECT} {DEMO_CONDITION} — cleaned PSD (mastoid ref, 0.1–100 Hz IIR)')
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Band-epoch shapes ──────────────────────────────────────────────────────
if epochs_demo is not None:
    print('Frequency band epoch shapes:')
    for band, (lf, hf) in BANDS.items():
        ep = filter_band(epochs_demo, lf, hf)
        print(f'  {band:5s} ({lf:4.1f}–{hf:5.1f} Hz): {ep.get_data().shape}')

In [ ]:
# ── Power spectral density per band for a representative epoch ─────────────
if epochs_demo is not None:
    fig, axes = plt.subplots(1, len(BANDS), figsize=(16, 4), sharey=False)
    sfreq = epochs_demo.info['sfreq']
    colors = ['#4e79a7','#f28e2b','#59a14f','#e15759','#b07aa1']
    for ax, (band, (lf, hf)), col in zip(axes, BANDS.items(), colors):
        ep = filter_band(epochs_demo, lf, hf)
        d  = ep.get_data().mean(axis=(0, 1))  # avg over epochs & channels
        t  = np.arange(len(d)) / sfreq
        ax.plot(t, d * 1e6, color=col, linewidth=0.7)
        ax.set_title(f'{band}\n{lf}–{hf} Hz')
        ax.set_xlabel('Time (s)')
        ax.set_ylabel('µV')
    fig.suptitle(f'{DEMO_SUBJECT} – Filtered band signals (mean epoch × channel)',
                 fontweight='bold')
    plt.tight_layout()
    plt.show()

## 6 · Single-subject demo — M1_1 SSAEP (event-locked)
Shows event-locked epoching on a task condition for a migraine subject.

In [ ]:
DEMO_M = 'M1_1'
DEMO_C = 'ssaep'

t0 = time.time()
res_m, _, epochs_m = preprocess_paper(
    subjects[DEMO_M][DEMO_C], DEMO_M, DEMO_C, OUTPUT_DIR,
    overwrite=True, return_epochs=True,
)
print(f'Done in {time.time()-t0:.1f}s')
print(f"  Status  : {res_m['status']}")
if res_m['status'] == 'failed':
    print(f"  Error   : {res_m.get('error')}")
else:
    print(f"  Shape   : {res_m['epoch_shape']}   (n_epochs=100 task trials)")
    print(f"  Bad ch  : {res_m['n_bad_channels']}")
    print(f"  ICs out : {res_m['n_rejected_ics']}")
    print(f"  Events  : {res_m['n_task_events']}")

In [ ]:
# ── Event-related potential: grand mean over all 128 channels ─────────────
if epochs_m is not None and res_m['status'] == 'success':
    evoked = epochs_m.average()
    sfreq  = epochs_m.info['sfreq']
    t = evoked.times
    d = evoked.data.mean(axis=0) * 1e6   # mean over channels

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(t, d, linewidth=1.2, color='#4e79a7')
    ax.axvline(0, color='r', linestyle='--', alpha=0.7, label='Stimulus onset')
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('µV (mean across channels)')
    ax.set_title(f'{DEMO_M} – SSAEP event-locked ERP  (0–2 s post-stimulus, all trials)')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7 · Full batch preprocessing

Processes all 39 subjects × 3 conditions = **116 recordings** (M13_2 has no SSAEP → 115 actual files).  
The loop is **resumable** — existing `_broadband.npy` files are skipped.

In [ ]:
# ── (Optional) per-subject manual bad channel overrides ───────────────────
# Add entries here if visual inspection reveals specific bad channels.
# Example:  MANUAL_BADS = {'C3': ['AFF5h'], 'M7_1': ['PPO9h']}
MANUAL_BADS = {}

# ── Conditions to run ─────────────────────────────────────────────────────
CONDITIONS = ['resting', 'ssaep', 'ssvep']

# Set OVERWRITE=True to reprocess already-done files
OVERWRITE = False

In [ ]:
subjects = discover_subjects(DATASET_DIR)

tasks = [(sub, cond, cond_files[cond])
         for sub, cond_files in sorted(subjects.items())
         for cond in CONDITIONS
         if cond in cond_files]

print('='*70)
print('MIGRAINE HD-EEG — PAPER-FAITHFUL BATCH PREPROCESSING')
print('='*70)
print(f'Dataset dir  : {DATASET_DIR}')
print(f'Output dir   : {OUTPUT_DIR}')
print(f'Conditions   : {CONDITIONS}')
print(f'Total tasks  : {len(tasks)}')
print(f'Reference    : mastoid (M1+M2)')
print(f'Filter       : IIR Butterworth {HIGHPASS_FREQ}–{LOWPASS_FREQ} Hz')
print(f'Epochs       : {EPOCH_DURATION_S}s  |  reject >{AMPLITUDE_REJECT_UV*1e6:.0f}µV')
print(f'Resample     : {TARGET_SFREQ} Hz' if RESAMPLE else 'Resample     : NO (512 Hz kept)')
print('='*70)

In [ ]:
summary_path = os.path.join(OUTPUT_DIR, 'preprocessing_summary.csv')
all_results  = []
if os.path.exists(summary_path):
    all_results = pd.read_csv(summary_path).to_dict('records')

t_batch = time.time()

for i, (sub, cond, bdf) in enumerate(tasks, 1):
    bb_check = os.path.join(OUTPUT_DIR, cond, f'{sub}_broadband.npy')
    if os.path.exists(bb_check) and not OVERWRITE:
        print(f'[{i:3d}/{len(tasks)}] {sub:7s} {cond:7s} — SKIP (exists)')
        continue

    print(f'[{i:3d}/{len(tasks)}] {sub:7s} {cond:7s} — processing...', end=' ', flush=True)
    t0  = time.time()
    res = preprocess_paper(
        bdf, sub, cond, OUTPUT_DIR,
        manual_bads=MANUAL_BADS.get(sub),
        overwrite=OVERWRITE,
    )
    res['elapsed_s'] = round(time.time() - t0, 1)

    # Replace any prior row for this (subject, condition)
    all_results = [r for r in all_results
                   if not (r.get('subject') == sub and r.get('condition') == cond)]
    all_results.append(res)

    if res['status'] == 'success':
        shp = res['epoch_shape']
        print(f"done {res['elapsed_s']:.0f}s | {shp[0]} epochs × {shp[1]} ch × {shp[2]} samp "
              f"| {res['n_bad_channels']} bad ch | {res['n_rejected_ics']} ICs")
    elif res['status'] == 'skipped_existing':
        print('skipped')
    else:
        print(f"FAILED: {res.get('error')}")

    # Persist after every recording → safe to interrupt
    pd.DataFrame(all_results).to_csv(summary_path, index=False)

print(f'\nBatch finished in {(time.time()-t_batch)/60:.1f} min')
print(f'Summary → {summary_path}')

## 8 · QC summary & labels

In [ ]:
df = pd.read_csv(summary_path)
ok = df[df.status == 'success']

print(f'Successfully preprocessed : {len(ok)} / {len(df)} recordings')
print(f'Failures                  : {(df.status == "failed").sum()}')

print('\n── Bad channels (mean ± std per condition) ──')
print(ok.groupby('condition')['n_bad_channels']
        .agg(['mean','std','max']).round(1).to_string())

print('\n── ICA rejections (mean ± std per condition) ──')
print(ok.groupby('condition')['n_rejected_ics']
        .agg(['mean','std','max']).round(1).to_string())

print('\n── Epochs kept per group+condition ──')
print(ok.groupby(['group','condition'])['n_epochs_after']
        .agg(['mean','min','max']).round(0).to_string())

In [ ]:
# ── Bar chart: epochs per subject per condition ────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=False)
colors_map = {'migraine': '#e15759', 'control': '#4e79a7'}

for ax, cond in zip(axes, ['resting', 'ssaep', 'ssvep']):
    sub_df = ok[ok.condition == cond].sort_values(['group', 'subject'])
    bar_colors = [colors_map.get(g, 'grey') for g in sub_df['group']]
    ax.bar(sub_df['subject'], sub_df['n_epochs_after'], color=bar_colors, edgecolor='none')
    ax.set_xticklabels(sub_df['subject'], rotation=90, fontsize=7)
    ax.set_title(cond.upper(), fontweight='bold')
    ax.set_ylabel('Clean epochs')

from matplotlib.patches import Patch
axes[0].legend(handles=[Patch(facecolor='#e15759', label='Migraine'),
                         Patch(facecolor='#4e79a7', label='Control')])
fig.suptitle('Clean epochs per subject (paper-faithful preprocessing)', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Write labels.csv ──────────────────────────────────────────────────────
label_rows = [
    dict(subject=r['subject'], group=r['group'], label=r['label'],
         aura=r['aura'], condition=r['condition'],
         excluded_in_orig_study=r.get('excluded_in_orig_study', False),
         n_epochs=r.get('n_epochs_after'),
         npy_broadband=r.get('npy_broadband'),
         band_paths=str(r.get('band_paths', '')))
    for r in all_results if r.get('status') == 'success'
]
labels_path = os.path.join(OUTPUT_DIR, 'labels.csv')
pd.DataFrame(label_rows).to_csv(labels_path, index=False)
print(f'Labels saved → {labels_path}')
pd.DataFrame(label_rows)[['subject','group','label','aura','condition','n_epochs']].head(12)

## 9 · Build model-ready stacked datasets

Stacks all subjects per condition into `X` (broadband) + per-band `X_delta` … `X_gamma`, labels `y`, and `groups` (subject ids for `GroupKFold`).

In [ ]:
def build_stacked_dataset(output_dir, condition):
    labels_csv = os.path.join(output_dir, 'labels.csv')
    df = pd.read_csv(labels_csv)
    df = df[df.condition == condition].reset_index(drop=True)
    if df.empty:
        print(f'No rows for condition "{condition}"'); return

    bb_list   = []
    band_lists = {b: [] for b in BANDS}
    y_list, grp_list = [], []

    for _, row in df.iterrows():
        X = np.load(row['npy_broadband'])                 # (n, 128, T)
        n = X.shape[0]
        bb_list.append(X)
        y_list.append(np.full(n, row['label'], dtype=np.int64))
        grp_list.append(np.array([row['subject']] * n))

        cond_dir = os.path.join(output_dir, condition)
        for band in BANDS:
            bp = os.path.join(cond_dir, f"{row['subject']}_{band}.npy")
            if os.path.exists(bp):
                band_lists[band].append(np.load(bp))

    X_bb  = np.concatenate(bb_list).astype(np.float32)
    y     = np.concatenate(y_list)
    groups= np.concatenate(grp_list)
    ch_names = open(os.path.join(output_dir, 'channel_names.txt')).read().splitlines()

    save_dict = dict(X=X_bb, y=y, groups=groups,
                     ch_names=np.array(ch_names), sfreq=sfreq_out)
    for band, lst in band_lists.items():
        if lst:
            save_dict[f'X_{band}'] = np.concatenate(lst).astype(np.float32)

    out = os.path.join(output_dir, f'dataset_{condition}.npz')
    np.savez_compressed(out, **save_dict)

    mig = int((y == 1).sum()); ctrl = int((y == 0).sum())
    print(f'[{condition:7s}] X={X_bb.shape}  migraine={mig}  control={ctrl}  → {out}')


print('Building stacked datasets...')
for cond in CONDITIONS:
    try:
        build_stacked_dataset(OUTPUT_DIR, cond)
    except Exception as e:
        print(f'  {cond}: {e}')

## 10 · How to load for your model

```python
import numpy as np
from sklearn.model_selection import GroupKFold

d = np.load('data/MIGRAINE_paper_preprocessed/dataset_resting.npz', allow_pickle=True)

X       = d['X']        # (N, 128, 1024) broadband, float32, in Volts
X_alpha = d['X_alpha']  # (N, 128, 1024) alpha-filtered
y       = d['y']        # (N,)  0=control, 1=migraine
groups  = d['groups']   # (N,)  subject id strings  ← use with GroupKFold!
ch_names= d['ch_names'] # 128 channel names
sfreq   = float(d['sfreq'])  # 512 (or 256 if RESAMPLE=True)

# Subject-aware cross-validation (no data leakage)
cv = GroupKFold(n_splits=5)
for train_idx, test_idx in cv.split(X, y, groups):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
```

---

### Two datasets, two pipelines — what to use

| | `preprocess_migraine.py` | This notebook |
|---|---|---|
| Reference | Average | **Mastoid** |
| Epochs | 4 s, 50 % overlap | **2 s, no overlap / event-locked** |
| Consistent with LEMON | ✓ (combine datasets) | ✗ |
| Faithful to paper | ✗ | **✓** |
| More training epochs | ✓ | ✗ |
| Frequency bands saved | ✗ | **✓ (5 bands)** |

**Recommendation**: use both and compare model performance — the paper-faithful outputs are best for coherence-based features and replicating the original findings; the LEMON-style outputs give more data per subject and are better for deep learning.